In [1]:
import os

In [4]:
%pwd

'c:\\Users\\ASUS\\Desktop\\Personal_Project\\Kidney Disease Classification\\KIdney-Disease-Classification-System-using-CNN'

In [3]:
os.chdir("../")

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int

In [29]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
from cnnClassifier.entity.config_entity import PrepareBaseModelConfig
from pathlib import Path
class ConfigurationManager:
    '''this constructor when called reads about the file path from one folder ans creates the artifact 
    direcatry '''
    def __init__(
        
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        '''this function return the the model params and the paths of the model stored'''
        config = self.config.prepare_base_model
        create_directories([config.root_dir]) ## this creatas the root dirctary 

        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
        )
    
        return prepare_base_model_config

ImportError: cannot import name 'PrepareBaseModelConfig' from 'cnnClassifier.entity.config_entity' (c:\users\asus\desktop\personal_project\kidney disease classification\kidney-disease-classification-system-using-cnn\src\cnnClassifier\entity\config_entity.py)

In [12]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf

In [ ]:
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        '''Stores configuration'''
        self.config = config

    def get_base_model(self):## this function woul down load the base model and save it at the given location
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )

        self.save_model(path=self.config.base_model_path, model=self.model)

    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        '''Freeze layers'''

        if freeze_all:
            for layer in model.layers:
                layer.trainable = False

        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                layer.trainable = False   # ✅ FIXED

        '''Add classification head'''
        flatten_in = tf.keras.layers.Flatten()(model.output)

        prediction = tf.keras.layers.Dense(
            units=classes,
            activation="softmax"
        )(flatten_in)

        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )

        '''Compile model'''
        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        full_model.summary()
        return full_model## here the model is ready to use
    
    def update_base_model(self):
        if not hasattr(self, "model"):
            raise ValueError("Base model not found. Run get_base_model() first.")
    
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )
    
        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

In [27]:
from cnnClassifier import logger


In [23]:
try:
    logger.info("Initializing Configuration Manager")
    config = ConfigurationManager()
    logger.info("Fetching Prepare Base Model Config")
    prepare_base_model_config = config.get_prepare_base_model_config()
    logger.info("Initializing Prepare Base Model Component")
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    logger.info("Loading base model (VGG16)")
    prepare_base_model.get_base_model()
    logger.info("Updating base model with custom layers")
    prepare_base_model.update_base_model()
    logger.info("Base model prepared successfully")
except Exception as e:
    logger.error("Error in Prepare Base Model Pipeline")
    raise e

[2026-04-12 16:11:17,414: INFO: 1562325444: Initializing Configuration Manager]
[2026-04-12 16:11:17,417: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-12 16:11:17,418: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-12 16:11:17,418: INFO: common: created directory at: artifacts]
[2026-04-12 16:11:17,418: INFO: 1562325444: Fetching Prepare Base Model Config]
[2026-04-12 16:11:17,418: ERROR: 1562325444: Error in Prepare Base Model Pipeline]


AttributeError: 'ConfigurationManager' object has no attribute 'get_prepare_base_model_config'